# MediShield V3 — Record Linkage & Privacy Optimization (Google Colab CPU)

This notebook trains, calibrates, and audits **MediShield V3** on Google Colab CPU (no GPU needed):
1. Generates **50,000** diverse, shortcut-free record pairs (`35k` train / `7.5k` val / `7.5k` test) + **5,000** sealed challenge pairs with **zero identity overlap**.
2. Verifies zero data leakage and zero duplicate pairs.
3. Extracts 16 shared features (`features.py`) distinguishing **missing fields** from **contradictory evidence**.
4. Compares 5 lightweight models (Calibrated Logistic Regression, Random Forest, Gradient Boosting, HistGradientBoosting, Fellegi-Sunter baseline).
5. Calibrates 3-stage thresholds (`HIGH_CONFIDENCE_MATCH`, `UNCERTAIN_REVIEW`, `NON_MATCH`) on validation data.
6. Plots Precision-Recall curves, Confusion Matrices, and Missing-Data analysis, then exports V3 `.joblib` and `.json` artifacts.

### Cell 1 — Install Dependencies
Installs exact compatible versions of `scikit-learn`, `rapidfuzz`, `numpy`, `pandas`, `joblib`, and `matplotlib`.

In [ ]:
!pip -q install scikit-learn==1.6.1 rapidfuzz==3.11.0 numpy==2.2.1 pandas==2.2.3 joblib>=1.4.0 matplotlib>=3.9.0

### Cell 2 — Clone Repository
Clones `M20A03/Yugaantar` so Colab runs the exact same shared feature extractor (`features.py`) used by the offline FastAPI backend.

In [ ]:
import os
if not os.path.exists('Yugaantar'):
    !git clone https://github.com/M20A03/Yugaantar.git
%cd Yugaantar/medishield

### Cell 3 — Dataset Generation, Leakage Checks & Exploratory Data Analysis
Generates the V3 splits (`35k` train, `7.5k` val, `7.5k` test, `5k` sealed challenge) and verifies **0** overlapping names between splits.

In [ ]:
import sys
from collections import Counter
sys.path.insert(0, 'training')
sys.path.insert(0, 'backend')
from generate_datasets_v3 import generate_v3_splits

tr, va, te, sc = generate_v3_splits()
print('Sizes (Train/Val/Test/Sealed):', len(tr), len(va), len(te), len(sc))
print('Train families:', Counter(p['family'] for p in tr))
n_tr = {p['record_a']['name'] for p in tr} | {p['record_b']['name'] for p in tr}
n_sc = {p['record_a']['name'] for p in sc} | {p['record_b']['name'] for p in sc}
print('Train vs Sealed Name Overlap:', len(n_tr & n_sc))

### Cell 4 — Feature Engineering, 5-Model Comparison, Calibration & Sealed Evaluation
Runs `train_v3.py` to train all models, calibrate 3-stage thresholds on validation, benchmark end-to-end latency & PII pipeline, and export V3 artifacts.

In [ ]:
from train_v3 import run_v3_pipeline
report = run_v3_pipeline()

### Cell 5 — Precision-Recall Curve, Confusion Matrix & Missing-Data Analysis
Plots the validation vs. sealed challenge performance and prints the confusion matrix and review-set breakdown.

In [ ]:
import joblib, numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve
from app.services.record_linkage.features import extract_pair_features

bundle = joblib.load('backend/artifacts/linkage_model_v3.joblib')
X_sc = np.array([extract_pair_features(p['record_a'], p['record_b']) for p in sc], dtype=np.float32)
y_sc = np.array([p['label'] for p in sc])
probs = bundle['model'].predict_proba(X_sc)[:, 1]
prec, rec, _ = precision_recall_curve(y_sc, probs)

plt.figure(figsize=(6, 4))
plt.plot(rec, prec, linewidth=2)
plt.title('Sealed 5,000-Pair Challenge Precision-Recall Curve')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.grid(True, alpha=0.3)
plt.show()

print('Sealed 3-Stage Metrics:', report['sealed_5000_3stage'])

### Cell 6 — Download V3 Artifacts
Downloads `linkage_model_v3.joblib`, `pii_model_v3.joblib`, `model_metadata_v3.json`, `evaluation_report_v3.json`, `feature_schema.json`, and `threshold_config.json`.

In [ ]:
try:
    from google.colab import files
    for f in [
        'backend/artifacts/linkage_model_v3.joblib',
        'backend/artifacts/pii_model_v3.joblib',
        'backend/artifacts/model_metadata_v3.json',
        'backend/artifacts/evaluation_report_v3.json',
        'backend/artifacts/feature_schema.json',
        'backend/artifacts/threshold_config.json',
    ]:
        files.download(f)
except ImportError:
    print('Artifacts saved locally in backend/artifacts/')